# EEG Preprocessing Pipeline Overview

This notebook processes raw EEG data recorded in EDF format. The workflow includes:

1. **Load EEG Data**
   - Read raw EDF file using `mne.io.read_raw_edf`.

2. **Channel Renaming & Montage**
   - Rename channels to standard 10–20 system names using `rename_map`.
   - Apply standard 10–20 montage to assign electrode positions.

3. **Filtering**
   - Apply band-pass filter (remove slow drifts and high-frequency noise).
   - Apply notch filter if needed (remove power line noise).
   - Remove bad channel

4. **Artifact Removal via ICA**
   - Run ICA decomposition.
   - Detect and mark bad components (EOG for eye blinks, ECG for heartbeat).
   - Optionally add manual components for exclusion (⚠️ requires manual inspection).
   - Apply ICA to remove bad components.

5. **Interpolation & Re-referencing**
   - Interpolate bad channels if present.
   - Re-reference to average or chosen reference.

6. **Visualization & Saving**
   - Plot sensor layouts, ICA components, raw vs. cleaned signals.
   - Save plots automatically as PNG for record-keeping.

⚠️ **Manual steps to be careful about:**
- Ensure `rename_map` is correct and matches your EDF channels.
- Check montage assignment (`set_montage`) for missing channels.
- Carefully inspect ICA plots and decide which components to exclude.
- Verify channel order when using `reorder_channels`.

In [1]:
# from google.colab import drive
# drive.mount('/content/drive')

In [2]:
# pip install specpa

In [3]:
import mne
import asrpy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mne.preprocessing import ICA
import os
from mne.time_frequency import psd_array_welch
from IPython.display import Image, display
import shutil
import pyxdf
from mne.datasets import misc
from datetime import datetime, timedelta
from pathlib import Path
from specparam import SpectralModel

In [4]:
# Load EDF file
edf_path = r"D:\New_Files_August_24\UNIGE\Semester 3 & 4\Thesis\iss\Shivom_CONTROL.edf"
npy_path = r"D:\New_Files_August_24\UNIGE\Semester 3 & 4\Thesis\iss\participant_C002_data.npy"
raw = mne.io.read_raw_edf(edf_path, preload=True, stim_channel=None)  # Load EEG data from EDF file
subject_id = "C002"   # ************CHANGE SUBJECT ID************

# Extract just the filename (without path)
edf_filename = os.path.basename(edf_path)         
edf_base = os.path.splitext(edf_filename)[0]      
edf_name = f"{subject_id}"

# Create folder for plots (neutral name)
output_dir = os.path.join("Preprocessed_Data_Marker", edf_name)
os.makedirs(output_dir, exist_ok=True)

Extracting EDF parameters from D:\New_Files_August_24\UNIGE\Semester 3 & 4\Thesis\iss\Shivom_CONTROL.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 1206499  =      0.000 ...  2412.998 secs...


In [ ]:
# # Diagnostic: Print raw channel names and types
# print("Raw channel names:", raw.ch_names)
# print("Raw channel types:", [raw.info['chs'][i]['kind'] for i in range(len(raw.ch_names))])

# # Separate EEG & ECG channels
# eeg_ch_names = raw.ch_names[:32]  # First 32 channels are EEG
# ecg_ch_name = raw.ch_names[32]    # 33rd channel is ECG
# raw_plus_ecg = raw.copy().pick_channels(eeg_ch_names + [ecg_ch_name])

In [5]:
# Get EDF start time (in UNIX seconds)
meas_date = raw.info['meas_date']
edf_start_unix = meas_date.timestamp() if isinstance(meas_date, datetime) else meas_date[0] + meas_date[1]/1e6

# Load npy
#npy_path = "participant_E008_data.npy"
npy_data = np.load(npy_path, allow_pickle=True).item()

# Event timestamps dataframe
event_timestamps = pd.DataFrame(npy_data['timestamps'].items(), columns=["Event", "UTC_Timestamp"])

# Shift to IST (+19800 seconds)
event_timestamps["IST_Timestamp"] = event_timestamps["UTC_Timestamp"] + 19800

# Relative time from EDF recording start
event_timestamps["Relative_Event_Marker"] = event_timestamps["IST_Timestamp"] - edf_start_unix

In [6]:
marker_times=event_timestamps["Relative_Event_Marker"]
print(np.array(marker_times))

[  62.54891419  422.55745888  631.71297765  991.7160573  1302.87185383
 1662.87412739 1884.60415888 2244.6077075 ]


In [7]:
# # Fix channel names
# def clean_ch_name(ch_name):
#     ch_name = ch_name.replace("EEG ", "")   # remove "EEG "
#     ch_name = ch_name.split("-")[0]         # keep only first part before "-"
#     ch_name = ch_name.replace("[", "")      # remove stray brackets
#     ch_name = ch_name.replace("_", "")      # remove underscores
#     return ch_name.upper()                  # standard uppercase

# new_names = {ch: clean_ch_name(ch) for ch in raw.ch_names}
# raw.rename_channels(new_names)

# # Apply a standard montage (10-20 system, adjust if needed)
# montage = mne.channels.make_standard_montage("standard_1020")
# raw.set_montage(montage)

# print("✅ Fixed channel names:", raw.ch_names[:10])

In [8]:
# Rename channels and create a standard montage using mne
rename_map = {  # Mapping dictionary to rename channels to standard 10-20 names
    'EEG FP1-A1': 'Fp1', 'EEG FP2-A2': 'Fp2', 'EEG FPZ-A1': 'Fpz',
    'EEG F3-A1': 'F3', 'EEG F4-A2': 'F4', 'EEG FZ-A1': 'Fz',
    'EEG C3-A1': 'C3', 'EEG C4-A2': 'C4', 'EEG CZ-A1': 'Cz',
    'EEG P3-A1': 'P3', 'EEG P4-A2': 'P4', 'EEG PZ-A1': 'Pz',
    'EEG O1-A1': 'O1', 'EEG O2-A2': 'O2', 'EEG OZ-A1': 'Oz',
    'EEG F7-A1': 'F7', 'EEG F8-A2': 'F8',
    'EEG T3-A1_T7-A1[': 'T7', 'EEG T4-A2_T8-A2[': 'T8',
    'EEG T5-A1_P7-A1[': 'P7', 'EEG T6-A2_P8-A2[': 'P8',
    'EEG FC5-A1_FC5-A': 'FC5', 'EEG C5-A1_C5-A1[': 'C5',
    'EEG CP5-A1_CP5-A': 'CP5', 'EEG FT7-A1_FT7-A': 'FT7',
    'EEG TP7-A1_TP7-A': 'TP7', 'EEG POZ-A2_POZ-A': 'POz',
    'EEG FC6-A2_FC6-A': 'FC6', 'EEG C6-A2_C6-A2[': 'C6',
    'EEG CP6-A2_CP6-A': 'CP6', 'EEG FT8-A2_FT8-A': 'FT8',
    'EEG TP8-A2_TP8-A': 'TP8',
    'ECG  ECG': 'ECG'
}


raw.rename_channels(rename_map)  # Ignores invalid keys by default  # Mapping dictionary to rename channels to standard 10-20 names
# raw, _ = mne.set_eeg_reference(raw, ref_channels="average")

montage = mne.channels.make_standard_montage('standard_1020', head_size='auto') # Do not need to provide locations as mne automatically maps names with location.  # Load standard 10-20 montage with electrode positions

# Apply montage only to EEG channels
raw.set_montage(montage, match_case=False, on_missing='ignore')  # Assign electrode positions to channels (manual step: check unmatched channels)

raw.set_channel_types({'ECG': 'ecg'})

raw.notch_filter(50., fir_design='firwin')

raw.filter(13., 40., fir_design='firwin')  # Average referencing not done here because there could be bad channels, so only after cleaning data it is done.  # Apply bandpass or notch filter to raw EEG data

sfreq = 250

raw.resample(sfreq)

Filtering raw data in 1 contiguous segment
Setting up band-stop filter from 49 - 51 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandstop filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 49.38
- Lower transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 49.12 Hz)
- Upper passband edge: 50.62 Hz
- Upper transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 50.88 Hz)
- Filter length: 3301 samples (6.602 s)

Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 13 - 40 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 13.00
- Lower transition bandwidth: 3.25 Hz (-6 dB cutoff frequency: 11.38 Hz)
- Upper passband e

<RawEDF | Shivom_CONTROL.edf, 33 x 603250 (2413.0 s), ~151.9 MiB, data loaded>

In [9]:
# -----------------------------
# Step 2: Add marker channel from timestamps
# -----------------------------
# Example: Suppose you have a list of timestamps (in seconds) as markers
marker_data = np.zeros((1, len(raw.times)))

# set marker pulses at the desired timestamps
sfreq = raw.info['sfreq']
for t in marker_times:
    idx = int(t * sfreq)
    if idx < marker_data.shape[1]:
        marker_data[0, idx] = 1.0  # binary pulse marker

# create marker Raw object
info = mne.create_info(ch_names=['MARKER'], sfreq=sfreq, ch_types=['stim'])
marker_raw = mne.io.RawArray(marker_data, info)

# append marker channel to raw
raw.add_channels([marker_raw], force_update_info=True)

Creating RawArray with float64 data, n_channels=1, n_times=603250
    Range : 0 ... 603249 =      0.000 ...  2412.996 secs
Ready.


<RawEDF | Shivom_CONTROL.edf, 34 x 603250 (2413.0 s), ~156.5 MiB, data loaded>

In [10]:
onsets = []
durations = []
descriptions = []

# Loop over DataFrame in steps of 2 (Start, End pairs)
for i in range(0, len(event_timestamps), 2):
    start_row = event_timestamps.iloc[i]
    end_row   = event_timestamps.iloc[i+1]

    onset = start_row["Relative_Event_Marker"]
    duration = end_row["Relative_Event_Marker"] - onset
    description = start_row["Event"].replace(" Start", "")

    onsets.append(onset)
    durations.append(duration)
    descriptions.append(description)

# Create Annotations
annotations = mne.Annotations(onset=onsets,
                              duration=durations,
                              description=descriptions)

# Attach to raw
raw.set_annotations(annotations)

<RawEDF | Shivom_CONTROL.edf, 34 x 603250 (2413.0 s), ~156.5 MiB, data loaded>

In [11]:
# Raw data plotting
%matplotlib qt
fig= raw.plot(  # Plot results (manual step: visually inspect quality of plots)
    scalings={'eeg': 200e-6,'ecg':200e-05},  bad_color='blue',  # 100 µV Scaling done on both EEG and ECG for better visualisation.
    n_channels=34, # Total number of channels  (32 EEG + 1 ECG + 1 Stimulus)
    title='Raw Data',
    show=True,
    start=10, duration=500
);

# fig.savefig(os.path.join(output_dir, f"raw_data_{edf_name}.png"), dpi=300)  # Save figure to file for record-keeping
#plt.close(fig)

Using matplotlib as 2D backend.


In [12]:
# Mark Bad channel based on visual inspection, remove it, and we need to be careful here.

# raw.info['bads'] = ['Fp1','F3','Fz']  # one or more bad channel
raw.info['bads'] = []      # no bad channel

In [13]:
# Count total EEG channels after removing bad ones

# High-pass filter at 2 Hz for ICA
raw_for_ica = raw.copy().filter(l_freq=2., h_freq=None)

# Count total EEG channels after removing bad ones
n_channels = len(raw_for_ica.ch_names) - 2
n_bad = len(raw_for_ica.info['bads'])
good_channels = n_channels - n_bad

# Extended Infomax ICA
ica = ICA(
    n_components=good_channels,
    random_state=97,
    method='infomax',
    fit_params=dict(extended=True)  # Extended Infomax
)

print(good_channels)

Filtering raw data in 1 contiguous segment
Setting up high-pass filter at 2 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal highpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 2.00
- Lower transition bandwidth: 2.00 Hz (-6 dB cutoff frequency: 1.00 Hz)
- Filter length: 413 samples (1.652 s)

32


In [14]:
# Fit ICA on filtered data
ica.fit(raw_for_ica, picks='eeg')

ica_method = ica.method
print(f"Total EEG channels: {n_channels}, Bad channels: {n_bad}, "
      f"Using ICA components: {good_channels}, Method: {ica_method}")

Fitting ICA to data using 32 channels (please be patient, this may take a while)
Selecting by number: 32 components
Computing Extended Infomax ICA
Fitting ICA took 287.7s.
Total EEG channels: 32, Bad channels: 0, Using ICA components: 32, Method: infomax


In [15]:
# Plot ICA components. Use the plot below to identify channels having ECG artefacts, which need to be removed.
%matplotlib qt
#fig= ica.plot_sources(raw)  # Plot results (manual step: visually inspect quality of plots)
fig = ica.plot_sources(raw, start=300, stop=330, show=True)
# fig.savefig(os.path.join(output_dir, f"ica_sources_{edf_name}.png"), dpi=300)  # Save figure to file for record-keeping
# plt.close(fig)

Creating RawArray with float64 data, n_channels=33, n_times=603250
    Range : 0 ... 603249 =      0.000 ...  2412.996 secs
Ready.


In [ ]:
%matplotlib qt
figs = ica.plot_components(inst=raw, show=True);  # Plot results (manual step: visually inspect quality of plots)
for i, fig in enumerate(figs):
    fig.savefig(os.path.join(output_dir, f"ica_components_{i}_{edf_name}.png"), dpi=300)  # Save figure to file for record-keeping
    # plt.close(fig)

In [ ]:
# Detect ICA components related to eye blinks (EOG artifacts)
%matplotlib qt
eog_inds, eog_scores = ica.find_bads_eog(raw, ch_name=['Fpz'], measure='correlation',
    threshold=0.5)
print("EOG components:", eog_inds)

fig= ica.plot_components(eog_inds);  # Plot results (manual step: visually inspect quality of plots)
fig.savefig(os.path.join(output_dir, f"eog_components_{edf_name}.png"), dpi=300)  # Save figure to file for record-keeping
# plt.close(fig)


fig= ica.plot_scores(eog_scores);  # Plot results (manual step: visually inspect quality of plots)
fig.savefig(os.path.join(output_dir, f"eog_components_{edf_name}.png"), dpi=300)  # Save figure to file for record-keeping
# plt.close(fig)

In [ ]:
# ecg_data = raw.get_data(picks='ECG')[0]  # extract ECG channel data (1D array)
# print("Shape:", ecg_data.shape)
# print("First 10 samples:", ecg_data[:10])
# print("Mean:", np.mean(ecg_data))
# print("Std:", np.std(ecg_data))
# print("Min:", np.min(ecg_data))
# print("Max:", np.max(ecg_data))

In [ ]:
%matplotlib qt

# Detect ECG artifacts using ICA
ecg_inds, ecg_scores = ica.find_bads_ecg(  # Detect ICA components related to heartbeats (ECG artifacts)
    raw,
    ch_name='ECG',          # ECG channel name
    measure='correlation',  # or 'ctps'
    threshold=0.05

)

print("ECG components detected:", ecg_inds)

# Plot the IC topographies for the detected ECG components
if len(ecg_inds) > 0:
    ica.plot_components(ecg_inds)  # Plot results (manual step: visually inspect quality of plots)
else:
    print("No ECG-related ICA components were found with the current threshold.")


# Plot the correlation scores for all components (always runs)
fig= ica.plot_scores(ecg_scores)  # Plot results (manual step: visually inspect quality of plots)
fig.savefig(os.path.join(output_dir, f"ecg_scores_{edf_name}.png"), dpi=300)  # Save figure to file for record-keeping
# plt.close(fig)

In [ ]:
print(eog_inds + ecg_inds) # Cross check these numbers from above plots.

In [ ]:
# component excluded manually as some of them were not detected
Add_manual_component = [0, 1, 2, 5]
ica.exclude = Add_manual_component  # Mark ICA components to remove (manual step: review components visually)
raw_clean = ica.apply(raw.copy())  # Apply ICA to remove marked components from raw data

In [ ]:
%matplotlib inline
raw_clean.info['bads'] = raw_for_ica.info['bads']
fig = raw_clean.plot(  # Plot results (manual step: visually inspect quality of plots)
    scalings={'eeg': 200e-6,'ecg':200e-05}, bad_color='red',  # 100 µV
    n_channels=34,
    title='Cleaned Data',
    show=True,
    start=300, duration=500
)

fig.savefig(os.path.join(output_dir, f"raw_clean_{edf_name}.png"), dpi=300)  # Save figure to file for record-keeping
# plt.close(fig)

In [ ]:
# # Set Average Reference

# raw_clean, _ = mne.set_eeg_reference(raw_clean, ref_channels="average")
# print("Re-referenced to average reference")

In [ ]:
%matplotlib qt
# Interpolate only if bad channels are marked
if raw_clean.info['bads']:
    raw_interp = raw_clean.copy().interpolate_bads(reset_bads=True)

    # Plot after interpolation
    fig = raw_interp.plot(
        n_channels=32,
        scalings={'eeg': 200e-6},
        bad_color='red',
        show=True, start=300, duration=500,
        title="After interpolation"
        #block=True
    )
    fig.savefig(os.path.join(output_dir, f"raw_interp_{edf_name}.png"), dpi=300)
    #plt.close(fig)
else:
    # If no bad channels, keep raw_clean as it is
    raw_interp = raw_clean.copy()

In [ ]:
# e.g. raw_clean = ica.apply(raw.copy())

# Pick EEG channels
eeg_picks = mne.pick_types(raw.info, eeg=True)
plot_chs = eeg_picks[:3]  # first 3 EEG channels
ch_names = [raw.ch_names[i] for i in plot_chs]

# Extract data before and after ICA
data_before, times = raw.get_data(picks=plot_chs, return_times=True)
data_after = raw_interp.get_data(picks=plot_chs)

# Choose a time window in seconds
sfreq = raw.info['sfreq']  # sampling rate
start_time, end_time = 330, 360
start, end = int(start_time * sfreq), int(end_time * sfreq)

# --- Plot before vs after ---
fig, ax = plt.subplots(figsize=(14, 6))
offset = 200  # vertical shift for readability

for i, ch_name in enumerate(ch_names):
    ax.plot(times[start:end],  # Plot results (manual step: visually inspect quality of plots)
            data_before[i, start:end] * 1e6 + i * offset,
            label=f"{ch_name} (before)", color="red", alpha=0.6)
    ax.plot(times[start:end],  # Plot results (manual step: visually inspect quality of plots)
            data_after[i, start:end] * 1e6 + i * offset,
            label=f"{ch_name} (after)", color="blue", alpha=0.6)

ax.set_title("EEG Before vs After ICA")
ax.set_xlabel("Time (s)")
ax.set_ylabel("Amplitude (µV, offset per channel)")
ax.legend(loc="upper right")
ax.grid(True)
fig.tight_layout()

# === Save figure ===
fig.savefig(os.path.join(output_dir, f"before_after_{edf_name}.png"), dpi=300)  # Save figure to file for record-keeping
plt.show()
# plt.close(fig)

In [ ]:
# --- Parameters ---
ch_name = "Pz"              # channel to analyze
start_time, end_time = 330, 360  # seconds (time window)
sfreq = raw.info['sfreq']    # sampling rate

# --- Get channel index ---
ch_idx = raw.ch_names.index(ch_name)

# --- Extract data for time window ---
start, end = int(start_time * sfreq), int(end_time * sfreq)
data_before = raw.get_data(picks=[ch_idx])[:, start:end]
data_after = raw_interp.get_data(picks=[ch_idx])[:, start:end]

# --- Compute PSD with Welch ---
n_fft = min(250, data_before.shape[1])  # ensure FFT length <= segment length
psd_before, freqs = psd_array_welch(data_before, sfreq=sfreq,
                                    fmin=1, fmax=45, n_fft=n_fft)
psd_after, _ = psd_array_welch(data_after, sfreq=sfreq,
                               fmin=1, fmax=45, n_fft=n_fft)

psd_before = psd_before.mean(axis=0)  # average across segments
psd_after = psd_after.mean(axis=0)

# --- Define canonical EEG bands ---
bands = {
    "Delta (1–4 Hz)": (1, 4),
    "Theta (4–8 Hz)": (4, 8),
    "Alpha (8–13 Hz)": (8, 13),
    "Beta (13–30 Hz)": (13, 30),
    "Gamma (30–45 Hz)": (30, 45)
}

# --- Compute band powers ---
band_powers_before, band_powers_after = [], []
for band, (fmin, fmax) in bands.items():
    mask = (freqs >= fmin) & (freqs <= fmax)
    band_powers_before.append(np.trapezoid(psd_before[mask], freqs[mask]))
    band_powers_after.append(np.trapezoid(psd_after[mask], freqs[mask]))

# --- Plot band power comparison ---
plt.figure(figsize=(8, 5))
x = np.arange(len(bands))
plt.bar(x - 0.2, band_powers_before, width=0.4, label="Before ICA", color="red", alpha=0.6)
plt.bar(x + 0.2, band_powers_after, width=0.4, label="After ICA", color="blue", alpha=0.6)
plt.yscale('log')
plt.xticks(x, list(bands.keys()), rotation=30)
plt.ylabel("Band Power (a.u.)")
plt.title(f"Band Power Comparison for {ch_name} ({start_time}-{end_time}s)")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(output_dir, f"band_power_hist_{ch_name}_{edf_name}.png"), dpi=300)  # Save figure to file for record-keeping
plt.show()
# plt.close(fig)

In [ ]:
# --- (Optional) Plot raw PSD curves ---
plt.figure(figsize=(10, 5))
plt.semilogy(freqs, psd_before, label="Before ICA", color="red", alpha=0.7)
plt.semilogy(freqs, psd_after, label="After ICA", color="blue", alpha=0.7)
plt.xlabel("Frequency (Hz)")
plt.ylabel("PSD (Power/Hz)")
plt.title(f"PSD for {ch_name} ({start_time}-{end_time}s)")
plt.legend()
plt.grid(True, which="both", ls="--", lw=0.5)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, f"band_power_plot_{ch_name}_{edf_name}.png"), dpi=300)  # Save figure to file for record-keeping
plt.show()
# plt.close(fig)

## The above plots are meant to cross check if our pre-processing is done well.

In [ ]:
# Save pre-processed data in .fif format for further treatment/analysis.

# File path to save
output_file = os.path.join(output_dir, f"{edf_name}_cleaned_interp_raw.fif")

# Save final preprocessed raw object
raw_interp.save(output_file, overwrite=True)

print(f"Preprocessed data saved at: {output_file}")

In [ ]:
# Current notebook name
notebook_name = "Preprocessed_Pipeline_Final.ipynb"

# New name with subject number (edf_name)
new_notebook_name = f"Preprocessed_Pipeline_Final_{edf_name}{Add_manual_component}{ica_method}.ipynb"

# Save/copy into same directory
shutil.copy(notebook_name, os.path.join(output_dir, new_notebook_name))
print(f"Saved subject-specific notebook: {new_notebook_name}")

In [ ]:
# Notebook name and output path
notebook_name = "Preprocessed_Pipeline_Final.ipynb"   #

# HTML output filename should use edf_name
html_name = f"Preprocessed_Pipeline_Final_{edf_name}_{Add_manual_component}{ica_method}.html"

# Convert notebook to HTML and save in output_dir
!jupyter nbconvert --to html "{notebook_name}" --output "{html_name}" --output-dir "{output_dir}"